In [11]:
#importing packages and env-var

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [12]:
spark = (( SparkSession
            .builder
            .appName('payments')
            .config('spark.jars', f'{sqlserverjar}' )
            .config('spark.executor.extraClassPath', f'{sqlserverjar}')
            .config('spark.driver.extraClassPath', f'{sqlserverjar}')
            .getOrCreate()
        ))

In [13]:
#load data
tmp_paymentframe = (( spark
                        .read
                        .format('jdbc')
                        .option('url', mssqlsource)
                        .option('driver', mssqldriver)
                        .option('user', mssqluser)
                        .option('password', mssqlpwd)
                        .option('query', 
                                
                                """ 
                                
                                    select
                                    
                                        paymentmethodid,
                                        paymentmethodname
                                        
                                    from application.paymentmethods
                                
                                """
                                )
                    )).load()

In [14]:
#rename column

paymentframe = (( tmp_paymentframe
                    .withColumnRenamed('paymentmethodid', 'paymentid')
                    .withColumnRenamed('paymentmethodname', 'methodname')

                ))

In [15]:
(( 
  paymentframe
    .write
      .format('jdbc')
      .option('url', mssql_dwdest)
      .option('driver', mssqldriver)
      .option('user', mssqluser)
      .option('password', mssqlpwd)
      .option('dbtable', 'dimension.paymentmethod')
    .save(mode='overwrite')
))